# 04 — Build the Retrieval Index

[Notebook guide](README.md) · [Project overview](../README.md) · [English paper](../docs/en/paper.pdf)

Convert leaflet PDFs, split by headings, and build the saved vector index.

| Item | Details |
| --- | --- |
| Inputs | Local leaflet PDFs under the configured Drive folder. |
| Outputs | Chunk pickle file, `index.faiss`, and `index.pkl`. |
| Requirements | Colab/Drive with pymupdf4llm, LangChain, sentence-transformers, and FAISS; a GPU is optional for embedding. |
| Configuration overview | `folder_path`, `chunks_path`, and `db_path`; heading levels #, ##, ###; multilingual embedding model; diagnostic retrieval k = 3. |

**Recorded experiment:** run code cells in their existing order. This presentation edit preserves executable code, parameters, prompts, data, and experimental outputs. Configuration is summarized here without moving assignments. Installation logs are omitted; execution counts belong to the original run. Serbian prompts, reference text, docstrings, and printed outputs remain verbatim.


## 1. Install dependencies

In [1]:
!pip install pymupdf4llm langchain-text-splitters

In [8]:
!pip install faiss-cpu sentence-transformers langchain-huggingface langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 2. Import libraries

In [1]:
from google.colab import drive
import os
import pymupdf4llm
from langchain_text_splitters import MarkdownHeaderTextSplitter
import pickle
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS



## 3. Mount Drive and locate source PDFs

In [2]:
# 1. Mount Google Drive
drive.mount('/content/drive')

# Source folder path
folder_path = "/content/drive/MyDrive/NLP_proyect/datasets/dataset_RAG"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4. Split documents by Markdown headings

Inspect the example chunk and processed-file counts before saving the chunk collection.

In [4]:
# 2. Define Markdown heading levels for splitting
headers_to_split_on = [
    ("#", "Naslov_1"),
    ("##", "Naslov_2"),
    ("###", "Naslov_3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

svi_chunkovi = []

In [5]:
# 3. Iterate over the PDF files
print("Započinjem obradu fajlova...")
for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
        file_path = os.path.join(folder_path, filename)

        # Convert PDF to Markdown, preserving Cyrillic and Latin text
        md_text = pymupdf4llm.to_markdown(file_path)

        # Split text by Markdown headings
        fajl_chunkovi = markdown_splitter.split_text(md_text)

        # Store the source filename in each chunk for later retrieval
        for chunk in fajl_chunkovi:
            chunk.metadata["izvor_fajl"] = filename
            svi_chunkovi.append(chunk)

print(f"\nUspešno obradjeno. Ukupno kreirano chunk-ova: {len(svi_chunkovi)}")


Započinjem obradu fajlova...

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR pro

In [6]:
# Display the first chunk as an example
if svi_chunkovi:
    print("\n--- PRIMER CHUNK-A ---")
    print(f"Metapodaci: {svi_chunkovi[0].metadata}")
    print(f"Tekst:\n{svi_chunkovi[0].page_content[:300]}...")


--- PRIMER CHUNK-A ---
Metapodaci: {'Naslov_2': '**ANALGIN**<sup>**®**</sup> **, 2,5 g/5 mL, раствор за инјекцију metamizol-natrijum**', 'izvor_fajl': '515-01-03530-21-001.pdf'}
Tekst:
**Пажљиво прочитајте ово упутство, пре него што почнете да примењујете овај лек, јер оно садржи информације које су важне за Вас.**  
- Упутство сачувајте. Може бити потребно да га поново прочитате.  
- Ако имате додатних питања, обратите се свом лекару, фармацеуту или медицинској сестри. Овај лек п...


In [7]:
# Collect unique source filenames from metadata
obradjeni_fajlovi = set(chunk.metadata.get("izvor_fajl") for chunk in svi_chunkovi)
svi_pdf_fajlovi = set(f for f in os.listdir(folder_path) if f.lower().endswith(".pdf"))

print(f"Ukupno PDF fajlova u folderu: {len(svi_pdf_fajlovi)}")
print(f"Uspešno obradjeno fajlova: {len(obradjeni_fajlovi)}")

neobradjeni = svi_pdf_fajlovi - obradjeni_fajlovi
if neobradjeni:
    print(f"Fajlovi koji NISU obradjeni:\n{neobradjeni}")
else:
    print("Svi fajlovi su uspešno prebačeni u chunk-ove!")

Ukupno PDF fajlova u folderu: 54
Uspešno obradjeno fajlova: 54
Svi fajlovi su uspešno prebačeni u chunk-ove!


In [8]:
chunks_path = "/content/drive/MyDrive/NLP_proyect/datasets/RAG_chunks/svi_chunkovi.pkl"

with open(chunks_path, "wb") as f:
    pickle.dump(svi_chunkovi, f)

print("Chunkovi uspešno sačuvani na Drive.")

Chunkovi uspešno sačuvani na Drive.


## 5. Build and save the FAISS index

In [10]:
chunks_path = "/content/drive/MyDrive/NLP_proyect/datasets/RAG_chunks/svi_chunkovi.pkl"

with open(chunks_path, "rb") as f:
    svi_chunkovi = pickle.load(f)

print(f"Učitano {len(svi_chunkovi)} chunkova sa Drive-a.")

Učitano 1769 chunkova sa Drive-a.


In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Use the multilingual sentence embedding model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

print("Generišem vektore i pravim bazu (ovo može potrajati par minuta)...")
vector_db = FAISS.from_documents(svi_chunkovi, embedding_model)

db_path = "/content/drive/MyDrive/NLP_proyect/vector_database"
vector_db.save_local(db_path)

print(f"Vektorska baza uspešno napravljena i sačuvana na: {db_path}")

/tmp/ipykernel_11832/3584876492.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generišem vektore i pravim bazu (ovo može potrajati par minuta)...
Vektorska baza uspešno napravljena i sačuvana na: /content/drive/MyDrive/NLP_proyect/vector_database


## 6. Inspect retrieval

The original example retrieves three chunks. This diagnostic differs from the application’s five-chunk configuration.

In [12]:
# 1. Load the saved index from Drive
db_path = "/content/drive/MyDrive/NLP_proyect/vector_database"
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

vector_db = FAISS.load_local(db_path, embedding_model, allow_dangerous_deserialization=True)

# 2. Set an example query
upit = "Koje su kontraindikacije za Rivotril lek?" # Choose a medicine represented in the database
rezultati = vector_db.similarity_search(upit, k=3)

# 3. Display the results
print(f"Pronađeno {len(rezultati)} relevantna isecak:\n")
for i, rez in enumerate(rezultati):
    print(f"--- REZULTAT {i+1} ---")
    print(f"Fajl: {rez.metadata.get('izvor_fajl')}")
    print(f"Naslov: {rez.metadata.get('Naslov_2', 'Nema naslova')}")
    print(f"Tekst:\n{rez.page_content[:400]}...\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Pronađeno 3 relevantna isecak:

--- REZULTAT 1 ---
Fajl: 515-01-01691-21-001.pdf
Naslov: Nema naslova
Tekst:
Na početku lečenja lekom Rivotril možete primetiti sledeće neželjene efekte:  
- osećaj pospanosti i umora,  
- vrtoglavica i ošamućenost,  
- slabost ili mlitavost mišića, ili grčevite pokrete (poremećaj koordinacije),  
- nestabilnost pri hodu.  
Ako primetite bilo koji od ovih efekata, obratite se Vašem lekaru. Vaš lekar će Vam pomoći tako što ćete dobiti manje doze leka Rivotril koje će se pos...

--- REZULTAT 2 ---
Fajl: 515-01-01691-21-001.pdf
Naslov: Nema naslova
Tekst:
Lek Rivotril sadrži aktivnu supstancu klonazepam, koja pripada grupi lekova koji se nazivaju benzodiazepini.  
Lek Rivotril se primenjuje za lečenje većine oblika epilepsije, prevashodno kao dodatna terapija (istovremeno sa nekim drugim lekom) ili u refraktornim slučajevima (teško lečivim, odnosno upornim oblicima bolesti), a posebno kod apsans napada, uključujući atipične apsans napade, _Lennox-G...

--- 

## Reading the recorded results

Saved outputs document the original run. This presentation edit does not rerun the notebook, change its artifacts, or claim that a fresh environment will reproduce the recorded results exactly.